<a href="https://colab.research.google.com/github/umyunsang/edu/blob/main/ComputerScience/03_ai-ml-data/quantum-ml/1.quantum-ml-overview/state-change-analysis/4_single_qubit_gate_effects.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 4. Single-Qubit Gate Effects Practice

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umyunsang/edu/blob/main/ComputerScience/03_ai-ml-data/quantum-ml/1.quantum-ml-overview/state-change-analysis/4_single_qubit_gate_effects.ipynb)

이 노트북은 `4-1.py`, `4-2.py`, `4-3.py`, `4-4.py`를 하나의 gate-effect 비교 실습으로 정리한 버전입니다.

**확인한 원본 코드**
- `4-1.py`: 아무 gate도 적용하지 않은 `|0>` 측정 결과를 확인합니다.
- `4-2.py`: `H` gate를 적용한 뒤 측정합니다.
- `4-3.py`: `X` gate를 적용한 뒤 측정합니다.
- `4-4.py`: `H` gate 뒤에 `X` gate를 적용한 뒤 측정합니다.

**학습 목표**
- `I`, `H`, `X`, `H -> X` 회로의 측정 분포를 비교합니다.
- `X` gate가 `|0>`을 `|1>`로 바꾸는지 확인합니다.
- Hadamard 이후의 측정 결과가 확률적으로 해석되어야 함을 확인합니다.


## Outline

1. 원본 파일과 gate 조합 확인
2. Qiskit 실행 준비
3. 네 가지 1-qubit 회로 구성
4. counts와 probability vector 비교
5. 결과 시각화
6. 연습 문제


## 1. Source Map

네 파일은 gate 조합만 바꾸어 같은 방식으로 측정합니다.


In [ ]:
SOURCE_CASES = {
    "4-1.py": "measure |0> without a gate",
    "4-2.py": "apply H, then measure",
    "4-3.py": "apply X, then measure",
    "4-4.py": "apply H and then X, then measure",
}

for source_name, description in SOURCE_CASES.items():
    print(f"{source_name}: {description}")


## 2. Imports

이 실습은 Qiskit과 Qiskit Aer가 필요합니다. 새 Colab 런타임에서 import가 실패하면 설치 줄의 주석을 해제해 한 번 실행한 뒤, 런타임을 다시 시작하고 이어서 실행합니다.


In [ ]:
# If Qiskit is missing in a fresh Colab runtime, uncomment and run once.
# %pip install -q qiskit qiskit-aer

import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

SHOTS = 1000
SEED_SIMULATOR = 4101


## 3. Build the Gate Circuits

원본 코드의 gate 순서를 그대로 유지해 네 회로를 만듭니다.


In [ ]:
def run_counts(circuit: QuantumCircuit, shots: int, seed: int) -> dict[str, int]:
    simulator = AerSimulator(seed_simulator=seed)
    job = simulator.run(circuit, shots=shots)
    result = job.result()
    return {state: int(count) for state, count in result.get_counts().items()}


def probability_vector(counts: dict[str, int], shots: int) -> list[float]:
    return [counts.get("0", 0) / shots, counts.get("1", 0) / shots]


In [ ]:
identity_circuit = QuantumCircuit(1)
identity_circuit.measure_all()

h_circuit = QuantumCircuit(1)
h_circuit.h(0)
h_circuit.measure_all()

x_circuit = QuantumCircuit(1)
x_circuit.x(0)
x_circuit.measure_all()

h_then_x_circuit = QuantumCircuit(1)
h_then_x_circuit.h(0)
h_then_x_circuit.x(0)
h_then_x_circuit.measure_all()

gate_experiments = [
    ("4-1.py", "I", identity_circuit),
    ("4-2.py", "H", h_circuit),
    ("4-3.py", "X", x_circuit),
    ("4-4.py", "H then X", h_then_x_circuit),
]

for source_name, label, circuit in gate_experiments:
    print()
    print(f"{source_name} - {label}")
    print(circuit.draw(output="text"))


## 4. Run and Compare

각 회로의 counts와 `[P(0), P(1)]`를 비교합니다.


In [ ]:
results = {}
for index, (source_name, label, circuit) in enumerate(gate_experiments):
    counts = run_counts(circuit, SHOTS, SEED_SIMULATOR + index)
    vector = probability_vector(counts, SHOTS)
    results[label] = (counts, vector)
    print(f"{source_name} ({label})")
    print("counts:", counts)
    print("[P(0), P(1)]:", vector)


## 5. Visualize Gate Effects

`I`는 거의 항상 `0`, `X`는 거의 항상 `1`을 만들고, `H`가 포함된 회로는 `0`과 `1`을 거의 반반으로 만듭니다.


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3), sharey=True)

for axis, (_, label, _) in zip(axes, gate_experiments):
    _, vector = results[label]
    axis.bar(["P(0)", "P(1)"], vector)
    axis.set_title(label)
    axis.set_ylim(0, 1)

axes[0].set_ylabel("probability")
fig.suptitle("Single-qubit gate measurement effects")
fig.tight_layout()
plt.show()


## 6. Interpretation

`H`와 `H then X`는 computational basis 측정에서 거의 같은 확률 분포를 보입니다. 실제로 `X|+> = |+>`이므로 이 두 회로는 같은 측정 분포를 만듭니다. 반면 `X then H`처럼 gate 순서를 바꾸면 측정 분포는 비슷해도 phase가 달라질 수 있으므로, 그런 차이는 statevector 분석으로 확인해야 합니다.


In [ ]:
h_vector = results["H"][1]
h_then_x_vector = results["H then X"][1]

print("H probability vector:       ", h_vector)
print("H then X probability vector:", h_then_x_vector)
print("absolute difference:        ", [abs(a - b) for a, b in zip(h_vector, h_then_x_vector)])


## 7. Exercise

`X` gate 뒤에 `H` gate를 적용한 `X then H` 회로를 추가해 보세요. `H then X`와 측정 분포가 같은지 비교합니다.


In [ ]:
exercise_circuit = QuantumCircuit(1)
exercise_circuit.x(0)
exercise_circuit.h(0)
exercise_circuit.measure_all()

exercise_counts = run_counts(exercise_circuit, SHOTS, SEED_SIMULATOR + 99)
exercise_vector = probability_vector(exercise_counts, SHOTS)

print(exercise_circuit.draw(output="text"))
print("counts:", exercise_counts)
print("[P(0), P(1)]:", exercise_vector)


## Pitfall and Extension

**주의할 점**: 측정 결과만 보면 서로 다른 회로가 비슷해 보일 수 있습니다. 측정은 quantum state의 모든 정보를 그대로 보여주는 과정이 아니라, 선택한 basis에서 확률적으로 읽는 과정입니다.

**확장 실습**: `qiskit.quantum_info.Statevector`를 사용해 측정 전 statevector를 비교해 보세요.
